In [21]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score, f1_score, confusion_matrix, precision_score, recall_score, classification_report,roc_auc_score,roc_curve,RocCurveDisplay
import os
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from tqdm import tqdm
os.chdir('/home/pgcp-ai/MachineLearning/Cases/Wisconsin/')

In [22]:
cancer = pd.read_csv("BreastCancer.csv",index_col='Code')
cancer

,Clump,UniCell_Size,Uni_CellShape,MargAdh,SEpith,BareN,BChromatin,NoemN,Mitoses,Class
Code,,,,,,,,,,
61634,5,4,3,1,2,2,2,3,1,Benign
63375,9,1,2,6,4,10,7,7,2,Malignant
76389,10,4,7,2,2,8,6,1,1,Malignant
95719,6,10,10,10,8,10,7,10,7,Malignant
128059,1,1,1,1,2,5,5,1,1,Benign
...,...,...,...,...,...,...,...,...,...,...
1369821,10,10,10,10,5,10,10,10,7,Malignant
1371026,5,10,10,10,4,10,5,6,3,Malignant
1371920,5,1,1,1,2,1,3,2,1,Benign


In [23]:
cancer.info()

<class 'pandas.core.frame.DataFrame'>
Index: 699 entries, 61634 to 13454352
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Clump          699 non-null    int64 
 1   UniCell_Size   699 non-null    int64 
 2   Uni_CellShape  699 non-null    int64 
 3   MargAdh        699 non-null    int64 
 4   SEpith         699 non-null    int64 
 5   BareN          699 non-null    int64 
 6   BChromatin     699 non-null    int64 
 7   NoemN          699 non-null    int64 
 8   Mitoses        699 non-null    int64 
 9   Class          699 non-null    object
dtypes: int64(9), object(1)
memory usage: 60.1+ KB


In [24]:
cancer.isna().sum()

Clump            0
UniCell_Size     0
Uni_CellShape    0
MargAdh          0
SEpith           0
BareN            0
BChromatin       0
NoemN            0
Mitoses          0
Class            0
dtype: int64

In [25]:
le = LabelEncoder()
cancer["Class"] = le.fit_transform(cancer["Class"])

In [26]:
X, y = cancer.drop("Class", axis = 1), cancer["Class"]

In [27]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,stratify=y,random_state=26)

In [28]:
# ss = StandardScaler()
# X_train = ss.fit_transform(X_train)
# X_test = ss.transform(X_test)

In [29]:
svm = SVC(kernel='linear')
svm.fit(X_train,y_train)
y_pred = svm.predict(X_test)
accuracy_score(y_test,y_pred)

0.9523809523809523

In [30]:
Cs = np.linspace(0.001,5,20)
scores = []
for c in tqdm(Cs):
    svm = SVC(kernel='linear',C=c,probability=True,random_state=26)
    svm.fit(X_train,y_train)
    y_pred = svm.predict(X_test)
    acc = accuracy_score(y_test,y_pred)
    y_pred_prob = svm.predict_proba(X_test)
    log_loss_score = log_loss(y_test,y_pred_prob)
    scores.append([c,acc,log_loss_score])

100%|███████████████████████████████████████████| 20/20 [00:00<00:00, 25.27it/s]


In [31]:
df_scores = pd.DataFrame(scores,columns=['C','Accuracy Score','Log Loss'])
df_scores.sort_values('Log Loss',ascending=True)

,C,Accuracy Score,Log Loss
0,0.001000,0.961905,0.097227
1,0.264105,0.952381,0.102645
2,0.527211,0.952381,0.103446
4,1.053421,0.952381,0.103609
3,0.790316,0.952381,0.103611
5,1.316526,0.952381,0.103761
6,1.579632,0.952381,0.103895
9,2.368947,0.952381,0.103968
7,1.842737,0.952381,0.103985
8,2.105842,0.952381,0.103987


### To use `predict_proba` in SVC we have to first put `probability=True` and re-assign `random_state=26` in SVC Constructor to enable `predict_proba` outputs and avoid errors

In [32]:
svm = SVC(C=0.790316,kernel='linear',probability=True,random_state=26)
svm.fit(X_train,y_train)
y_pred_prob = svm.predict_proba(X_test)
log_loss(y_test,y_pred_prob)

0.10361056406343505

### Polynomial Kernel

In [33]:
svm = SVC(C=0.790316,kernel='poly', degree=5,probability=True,random_state=26)
svm.fit(X_train,y_train)
y_pred_prob = svm.predict_proba(X_test)
log_loss(y_test,y_pred_prob)

0.17137737687784116

In [34]:
Cs = np.linspace(0.001,5,20)
degree = [2,3,4]
scores = []
for c in tqdm(Cs):
    for d in degree: 
        svm = SVC(kernel='poly',C=c,probability=True,degree=d,random_state=26)
        svm.fit(X_train,y_train)
        y_pred = svm.predict(X_test)
        acc = accuracy_score(y_test,y_pred)
        y_pred_prob = svm.predict_proba(X_test)
        log_loss_score = log_loss(y_test,y_pred_prob)
        scores.append([c,d,acc,log_loss_score])

100%|███████████████████████████████████████████| 20/20 [00:01<00:00, 13.68it/s]


In [35]:
df_scores = pd.DataFrame(scores,columns=['C','Degree','Accuracy Score','Log Loss'])
df_scores.sort_values(['Log Loss','Accuracy Score'],ascending=[True,False])

,C,Degree,Accuracy Score,Log Loss
25,2.105842,3,0.961905,0.104768
28,2.368947,3,0.961905,0.104804
33,2.895158,2,0.957143,0.105757
22,1.842737,3,0.961905,0.105781
31,2.632053,3,0.961905,0.105833
30,2.632053,2,0.957143,0.105899
36,3.158263,2,0.957143,0.106061
27,2.368947,2,0.957143,0.106274
39,3.421368,2,0.957143,0.106466
45,3.947579,2,0.957143,0.106513


### Radial Kernel

In [36]:
Cs = np.linspace(0.001,5,20)
Gs = np.linspace(0.001, 5, 20)
scores = []
for c in tqdm(Cs):
    for g in Gs: 
        svm = SVC(kernel='rbf',C=c,probability=True,gamma = g,random_state=26)
        svm.fit(X_train,y_train)
        y_pred = svm.predict(X_test)
        acc = accuracy_score(y_test,y_pred)
        y_pred_prob = svm.predict_proba(X_test)
        log_loss_score = log_loss(y_test,y_pred_prob)
        scores.append([c,g,acc,log_loss_score])

100%|███████████████████████████████████████████| 20/20 [00:24<00:00,  1.20s/it]


In [37]:
df_scores = pd.DataFrame(scores, columns = ['C', 'G', 'Accuracy Score', 'Log Loss'])

In [38]:
df_scores.sort_values(['Log Loss', 'Accuracy Score'], ascending = [True, False])

,C,G,Accuracy Score,Log Loss
260,3.421368,0.001000,0.957143,0.097582
80,1.053421,0.001000,0.957143,0.097644
140,1.842737,0.001000,0.957143,0.098023
40,0.527211,0.001000,0.961905,0.098143
280,3.684474,0.001000,0.957143,0.098186
...,...,...,...,...
15,0.001000,3.947579,0.657143,0.577960
16,0.001000,4.210684,0.657143,0.584281
17,0.001000,4.473789,0.657143,0.588494
19,0.001000,5.000000,0.657143,0.589095


### Sigmoid Kernel

In [39]:
Cs = np.linspace(0.001,5,20)
coef = np.linspace(-5, 5, 20)
scores = []
for c in tqdm(Cs):
    for cf in coef: 
        svm = SVC(kernel='sigmoid',C=c,probability=True,coef0=cf,random_state=26)
        svm.fit(X_train,y_train)
        y_pred = svm.predict(X_test)
        acc = accuracy_score(y_test,y_pred)
        y_pred_prob = svm.predict_proba(X_test)
        log_loss_score = log_loss(y_test,y_pred_prob)
        scores.append([c,cf,acc,log_loss_score])


100%|███████████████████████████████████████████| 20/20 [00:29<00:00,  1.49s/it]


In [40]:
df_scores = pd.DataFrame(scores, columns = ['C', 'coef0', 'Accuracy', 'Log Loss'])
df_scores.sort_values(['Log Loss', 'Accuracy'], ascending = [True, False])

,C,coef0,Accuracy,Log Loss
22,0.264105,-3.947368,0.952381,0.146260
41,0.527211,-4.473684,0.942857,0.152380
5,0.001000,-2.368421,0.657143,0.154060
100,1.316526,-5.000000,0.942857,0.157853
80,1.053421,-5.000000,0.947619,0.160801
...,...,...,...,...
327,4.210684,-1.315789,0.623810,0.627890
347,4.473789,-1.315789,0.623810,0.627996
367,4.736895,-1.315789,0.623810,0.628090
387,5.000000,-1.315789,0.623810,0.628175


In [41]:
print(np.linspace(1,10,5))

[ 1.    3.25  5.5   7.75 10.  ]
